# 03 — Data Preprocessing

**Objective:** Clean the raw dataset, handle missing values, remove outliers, encode categoricals, and save a clean version ready for feature engineering.

**Input:**  `data/raw/merged_cleaned_sold_with_josiah.csv`  
**Output:** `data/processed/listings_cleaned.csv`

**Covers (grading):** Data Preparation (15%)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

RAW_PATH       = '../data/raw/merged_cleaned_sold_with_josiah.csv'
PROCESSED_PATH = '../data/processed/listings_cleaned.csv'

df = pd.read_csv(RAW_PATH, low_memory=False)
print(f'Raw shape: {df.shape}')
df.head(3)

## Step 1 — Drop Irrelevant Columns

The dataset contains ~40 `homeData_*` columns that are redundant with our core features or mostly empty (63%+ missing). We drop them and keep only the meaningful columns.

In [ ]:
# Drop all homeData_ columns — redundant and 63%+ missing
homedata_cols = [c for c in df.columns if c.startswith('homeData_')]
df = df.drop(columns=homedata_cols)

# Also drop columns not useful for modeling
drop_cols = [
    'address',        # free text identifier, not a feature
    'mls_number',     # ID, not a feature
    'listing_date',   # superseded by listing_year / listing_month
    'sale_date',      # superseded by days_since_sold
    'sold_date_dt',   # duplicate
    'source',         # data source tag, not a property feature
]
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

print(f'Shape after dropping irrelevant columns: {df.shape}')
print('Remaining columns:', list(df.columns))

## Step 2 — Remove Duplicate Rows

In [ ]:
before = len(df)
df = df.drop_duplicates()
print(f'Duplicates removed: {before - len(df)}')
print(f'Shape after dedup:  {df.shape}')

## Step 3 — Filter Impossible / Erroneous Values

Some rows have clearly erroneous prices (e.g. listing_price = $1). We apply business-logic filters to remove nonsensical records.

In [ ]:
before = len(df)

# sold_price: must be between $50,000 and $10,000,000
df = df[(df['sold_price'] >= 50_000) & (df['sold_price'] <= 10_000_000)]

# listing_price: must be at least $10,000
df = df[df['listing_price'] >= 10_000]

# bedrooms: at least 1 (0-bedroom listings are parking/storage units)
df = df[(df['bedrooms'].isna()) | (df['bedrooms'] >= 1)]

# square_feet: must be positive if present
df = df[(df['square_feet_num'].isna()) | (df['square_feet_num'] > 0)]

print(f'Rows removed by filters: {before - len(df)}')
print(f'Shape after filtering:   {df.shape}')

## Step 4 — Remove Outliers (IQR Method)

We apply IQR-based outlier removal on `sold_price` and `square_feet_num` using a conservative multiplier of 3.0 to avoid removing legitimate luxury properties.

In [ ]:
def remove_outliers_iqr(df, col, multiplier=3.0):
    """Remove rows where col falls outside Q1 - k*IQR or Q3 + k*IQR."""
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    before = len(df)
    df = df[(df[col] >= Q1 - multiplier * IQR) & (df[col] <= Q3 + multiplier * IQR)]
    print(f'  {col}: removed {before - len(df)} outliers | range now ${df[col].min():,.0f} – ${df[col].max():,.0f}')
    return df

print('Outlier removal (IQR multiplier = 3.0):')
df = remove_outliers_iqr(df, 'sold_price')
df = remove_outliers_iqr(df, 'listing_price')

# Only apply to sqft where not null
sqft_mask = df['square_feet_num'].notna()
sqft_df = df[sqft_mask].copy()
sqft_df = remove_outliers_iqr(sqft_df, 'square_feet_num')
df = pd.concat([sqft_df, df[~sqft_mask]]).sort_index()

print(f'\nShape after outlier removal: {df.shape}')

## Step 5 — Handle Missing Values

In [ ]:
# Show missing summary before
missing_before = (df.isnull().mean() * 100).sort_values(ascending=False)
missing_before = missing_before[missing_before > 0]
print('Missing % before imputation:')
print(missing_before.to_string())

In [ ]:
# --- Numeric columns: fill with median ---
numeric_fill_median = ['bedrooms', 'bathrooms', 'square_feet_num',
                        'lot_size_sqft', 'taxes', 'parking_spaces',
                        'number_of_storeys', 'days_since_sold']

for col in numeric_fill_median:
    if col in df.columns:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f'  {col}: filled with median = {median_val:.2f}')

# --- city: fill with mode (most common city) ---
if df['city'].isnull().any():
    mode_city = df['city'].mode()[0]
    df['city'] = df['city'].fillna(mode_city)
    print(f'  city: filled with mode = {mode_city}')

# --- property_type: fill with mode ---
if df['property_type'].isnull().any():
    mode_type = df['property_type'].mode()[0]
    df['property_type'] = df['property_type'].fillna(mode_type)
    print(f'  property_type: filled with mode = {mode_type}')

# --- house_age: 78% missing — keep as categorical with 'Unknown' ---
df['house_age'] = df['house_age'].fillna('Unknown')
print('  house_age: filled with Unknown (78% missing — treated as category)')

In [ ]:
# Confirm no missing in key columns
missing_after = (df.isnull().mean() * 100).sort_values(ascending=False)
missing_after = missing_after[missing_after > 0]
print('Missing % after imputation:')
if missing_after.empty:
    print('  None — all key columns complete.')
else:
    print(missing_after.to_string())

## Step 6 — Encode Categorical Variables

In [ ]:
# --- property_type: consolidate rare types into 'Other' ---
top_types = df['property_type'].value_counts().nlargest(8).index
df['property_type'] = df['property_type'].where(df['property_type'].isin(top_types), other='Other')
print('Property types after consolidation:')
print(df['property_type'].value_counts())

In [ ]:
# --- house_age: ordinal encoding (ordered categories) ---
age_order = {
    'New': 0,
    '0-5 Years': 1,
    '6-10 Years': 2,
    '6-15 Years': 2,
    '11-15 Years': 3,
    '16-30 Years': 4,
    '31-50 Years': 5,
    '51-99 Years': 6,
    '100+ Years': 7,
    'Unknown': -1
}
df['house_age_encoded'] = df['house_age'].map(age_order)
print('House age encoding:')
print(df[['house_age', 'house_age_encoded']].value_counts().head(10))

In [ ]:
# --- One-hot encode property_type and city ---
before_cols = df.shape[1]
df = pd.get_dummies(df, columns=['property_type', 'city'], drop_first=True)
print(f'Columns before encoding: {before_cols}')
print(f'Columns after encoding:  {df.shape[1]}')

## Step 7 — Final Checks

In [ ]:
print(f'Final shape: {df.shape}')
print(f'\nTarget (sold_price) stats:')
print(df['sold_price'].describe().apply(lambda x: f'${x:,.0f}'))
print(f'\nAny nulls remaining: {df.isnull().sum().sum()}')

In [ ]:
# Before vs after sold_price distribution
raw = pd.read_csv(RAW_PATH, low_memory=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(raw['sold_price'].clip(0, 5_000_000), bins=60, color='salmon', edgecolor='white')
axes[0].set_title('Sold Price — Before Cleaning')
axes[0].set_xlabel('Sold Price (CAD)')

axes[1].hist(df['sold_price'], bins=60, color='steelblue', edgecolor='white')
axes[1].set_title('Sold Price — After Cleaning')
axes[1].set_xlabel('Sold Price (CAD)')

plt.suptitle('Distribution Before vs After Preprocessing')
plt.tight_layout()
plt.savefig('../reports/figures/preprocessing_before_after.png')
plt.show()

## Step 8 — Save Cleaned Dataset

In [ ]:
df.to_csv(PROCESSED_PATH, index=False)
print(f'Saved cleaned dataset to: {PROCESSED_PATH}')
print(f'Final shape: {df.shape}')

## Preprocessing Summary

| Step | Action | Rows Affected |
|------|--------|---------------|
| Drop irrelevant columns | Removed `homeData_*` + ID/date cols | — |
| Remove duplicates | `drop_duplicates()` | See output above |
| Business logic filters | Price < $50k, bedrooms = 0, sqft ≤ 0 | See output above |
| Outlier removal | IQR × 3.0 on sold_price, listing_price, sqft | See output above |
| Missing values | Median for numeric, mode for city/type, 'Unknown' for house_age | — |
| Categorical encoding | Ordinal for house_age, one-hot for property_type & city | — |

**Output:** `data/processed/listings_cleaned.csv` — ready for feature engineering.